In [28]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

In [29]:
# Read the raw scraped data
# If the file doesn't exist, run the scraping notebook first
try:
    df = pd.read_csv("ufc_fighters_raw.csv")
    print(f"Loaded {len(df)} fighters from ufc_fighters_raw.csv")
except FileNotFoundError:
    print("Error: ufc_fighters_raw.csv not found. Please run UFC_scraping.ipynb first.")
    df = pd.DataFrame()  # Create empty dataframe to prevent errors 


Loaded 4444 fighters from ufc_fighters_raw.csv


In [30]:
# Parse record - handle edge cases like "NC)" or invalid formats
# First, filter out rows with invalid record formats
print(f"Total fighters before record parsing: {len(df)}")

# Check for invalid records
invalid_records = df['Record'].str.contains("NC\)|--|N/A", na=True, regex=True)
print(f"Found {invalid_records.sum()} fighters with invalid records (NC, --, etc.)")

# Only parse valid records (format: "W-L-D")
valid_mask = df['Record'].str.match(r'^\d+-\d+-\d+$', na=False)
print(f"Valid records: {valid_mask.sum()}")

# Initialize columns first
df['Wins'] = None
df['Losses'] = None
df['Draws'] = None

# Parse valid records - split and assign properly
if valid_mask.sum() > 0:
    split_records = df.loc[valid_mask, 'Record'].str.split("-", expand=True)
    df.loc[valid_mask, 'Wins'] = split_records[0]
    df.loc[valid_mask, 'Losses'] = split_records[1]
    df.loc[valid_mask, 'Draws'] = split_records[2]

print(f"Sample parsed records:")
print(df[['Record', 'Wins', 'Losses', 'Draws']].head(10))

Total fighters before record parsing: 4444
Found 529 fighters with invalid records (NC, --, etc.)
Valid records: 3915
Sample parsed records:
    Record  Wins Losses Draws
0    5-3-0     5      3     0
1    4-6-0     4      6     0
2   28-4-0    28      4     0
3    9-5-0     9      5     0
4  10-15-0    10     15     0
5      NC)  None   None  None
6    9-0-1     9      0     1
7   20-8-0    20      8     0
8      NC)  None   None  None
9    6-2-0     6      2     0


In [31]:
# Convert to int, handling NaN values
# Only convert valid records, keep invalid ones as NaN (they'll be filtered later)
df['Wins'] = pd.to_numeric(df['Wins'], errors='coerce')
df['Losses'] = pd.to_numeric(df['Losses'], errors='coerce')
df['Draws'] = pd.to_numeric(df['Draws'], errors='coerce')

# Fill NaN with 0 for calculation purposes, but we'll filter invalid records later
df['Wins'] = df['Wins'].fillna(0).astype(int)
df['Losses'] = df['Losses'].fillna(0).astype(int)
df['Draws'] = df['Draws'].fillna(0).astype(int)

print(f"Record parsing complete. Wins/Losses/Draws columns created.")
print(f"Sample values:")
print(df[['Record', 'Wins', 'Losses', 'Draws']].head(10))

Record parsing complete. Wins/Losses/Draws columns created.
Sample values:
    Record  Wins  Losses  Draws
0    5-3-0     5       3      0
1    4-6-0     4       6      0
2   28-4-0    28       4      0
3    9-5-0     9       5      0
4  10-15-0    10      15      0
5      NC)     0       0      0
6    9-0-1     9       0      1
7   20-8-0    20       8      0
8      NC)     0       0      0
9    6-2-0     6       2      0


In [32]:
# Calculate win percentage, handling division by zero
total_fights = df['Wins'] + df['Losses'] + df['Draws']
print(f"Total fights calculation:")
print(f"  Fighters with Wins > 0: {(df['Wins'] > 0).sum()}")
print(f"  Fighters with Losses > 0: {(df['Losses'] > 0).sum()}")
print(f"  Fighters with total fights > 0: {(total_fights > 0).sum()}")
print(f"  Fighters with total fights = 0: {(total_fights == 0).sum()}")

# Calculate win percentage only for fighters with fights
df['Win Percentage'] = np.where(
    total_fights > 0,
    round(df['Wins'] / total_fights, 2),
    0.0
)

print(f"\nWin percentage calculated.")
print(f"Sample win percentages:")
print(df[['Name', 'Record', 'Wins', 'Losses', 'Draws', 'Win Percentage']].head(10))

Total fights calculation:
  Fighters with Wins > 0: 3766
  Fighters with Losses > 0: 3750
  Fighters with total fights > 0: 3896
  Fighters with total fights = 0: 548

Win percentage calculated.
Sample win percentages:
                  Name   Record  Wins  Losses  Draws  Win Percentage
0            Tom Aaron    5-3-0     5       3      0            0.62
1         Danny Abbadi    4-6-0     4       6      0            0.40
2      Nariman Abbasov   28-4-0    28       4      0            0.88
3         Darion Abbey    9-5-0     9       5      0            0.64
4         David Abbott  10-15-0    10      15      0            0.40
5     Hamdy Abdelwahab      NC)     0       0      0            0.00
6   Mansur Abdul-Malik    9-0-1     9       0      1            0.90
7  Shamil Abdurakhimov   20-8-0    20       8      0            0.71
8         Hiroyuki Abe      NC)     0       0      0            0.00
9           Daichi Abe    6-2-0     6       2      0            0.75


In [33]:
# Filter out rows with missing critical data
# Use .str accessor safely with na=False parameter
if 'Height:' in df.columns:
    df = df[~df['Height:'].str.contains("--", na=False)]
if 'Reach:' in df.columns:
    df = df[~df['Reach:'].str.contains("--", na=False)]
if 'DOB:' in df.columns:
    df = df[~df['DOB:'].str.contains("--", na=False)]
if 'Str. Acc.:' in df.columns:
    df = df[~df['Str. Acc.:'].str.contains(" ", na=False)]

In [34]:
df['Nickname'] = df['Nickname'].fillna('No Name')

In [35]:
len(df.Nickname)

4444

In [36]:
df.head()

,Name,Record,Nickname,Weight:,DOB:,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,STANCE:,Reach:,Wins,Losses,Draws,Win Percentage
0,Tom Aaron,5-3-0,No Name,155 lbs.,"Jul 13, 1978",0.00,0%,0.00,0%,0.00,0%,0%,0.0,NaN,NaN,NaN,5,3,0,0.62
1,Danny Abbadi,4-6-0,The Assassin,155 lbs.,"Jul 03, 1983",3.29,38%,4.41,57%,0.00,0%,77%,0.0,"5' 11""",Orthodox,NaN,4,6,0,0.40
2,Nariman Abbasov,28-4-0,Bayraktar,155 lbs.,"Feb 01, 1994",3.00,20%,5.67,46%,0.00,0%,66%,0.0,"5' 8""",Orthodox,"66""",28,4,0,0.88
3,Darion Abbey,9-5-0,No Name,265 lbs.,"Feb 25, 1993",8.44,50%,14.06,28%,0.00,0%,0%,0.0,"6' 2""",Orthodox,"80""",9,5,0,0.64
4,David Abbott,10-15-0,Tank,265 lbs.,"Apr 26, 1965",1.35,30%,3.55,38%,1.07,33%,66%,0.0,"6' 0""",Switch,NaN,10,15,0,0.40


In [37]:
df.columns

Index(['Name', 'Record', 'Nickname', 'Weight:', 'DOB:', 'SLpM:', 'Str. Acc.:',
       'SApM:', 'Str. Def:', 'TD Avg.:', 'TD Acc.:', 'TD Def.:', 'Sub. Avg.:',
       'Height:', 'STANCE:', 'Reach:', 'Wins', 'Losses', 'Draws',
       'Win Percentage'],
      dtype='object')

In [38]:
# Parse height (format: "5' 11\"")
if 'Height:' in df.columns:
    # Only split non-NaN values
    height_mask = df['Height:'].notna()
    if height_mask.sum() > 0:
        df.loc[height_mask, ['Feet','Inches']] = df.loc[height_mask, 'Height:'].str.split("'", expand=True)
    
# Parse reach (format: "72\"")
if 'Reach:' in df.columns:
    # Handle NaN values - convert to numeric first, then to int (NaN will become 0, but we'll filter later)
    reach_str = df['Reach:'].str.slice(0,-1)
    df['Reach:'] = pd.to_numeric(reach_str, errors='coerce').fillna(0).astype(int)
    # Note: Rows with NaN reach will be filtered out in the next cell
    
# Parse weight (format: "170 lbs.")
if 'Weight:' in df.columns:
    # Handle NaN values - convert to numeric first, then to int
    weight_str = df['Weight:'].str.slice(0,-5)
    df['Weight:'] = pd.to_numeric(weight_str, errors='coerce').fillna(0).astype(int)
    # Note: Rows with NaN weight will be filtered out in the next cell


In [39]:
# Convert height to total inches
if 'Inches' in df.columns:
    # Only process non-NaN values
    inches_mask = df['Inches'].notna()
    if inches_mask.sum() > 0:
        # Remove the quote from inches (e.g., " 11\"" -> " 11")
        df.loc[inches_mask, 'Inches'] = df.loc[inches_mask, 'Inches'].str.slice(0,-1)
        
        # Convert to numeric, handling NaN
        df['Feet'] = pd.to_numeric(df['Feet'], errors='coerce').fillna(0).astype(int)
        df['Inches'] = pd.to_numeric(df['Inches'], errors='coerce').fillna(0).astype(int)
        
        # Convert inches to feet (divide by 12) and add to feet
        df.loc[inches_mask, 'Inches'] = df.loc[inches_mask, 'Inches'] / 12
        df.loc[inches_mask, 'Height:'] = round(df.loc[inches_mask, 'Feet'] + df.loc[inches_mask, 'Inches'], 2)
    
    # For rows without height data, set Height: to NaN (will be filtered later)
    if 'Height:' not in df.columns:
        df['Height:'] = np.nan

In [40]:
# Drop unnecessary columns (only if they exist)
cols_to_drop = ['Feet', 'Inches', 'Record', 'Unnamed: 0', 'STANCE:']
cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(cols_to_drop, axis=1)

# Rename DOB to Age
if 'DOB:' in df.columns:
    df = df.rename(columns={'DOB:': "Age"})

In [41]:
df['Age'] = pd.to_datetime(df['Age'],format='%b %d, %Y')

In [42]:
# Calculate age in years
now = dt.today()

# Handle dates that might be in the future (subtract 100 years if date is after today)
# This can happen with data entry errors
future_mask = df['Age'] > now
if future_mask.sum() > 0:
    df.loc[future_mask, 'Age'] = df.loc[future_mask, 'Age'] - relativedelta(years=100)

# Calculate age as the difference in years
# Convert timedelta to years by dividing by 365.25 days (accounting for leap years)
df['Age'] = (now - df['Age']).dt.days / 365.25
df['Age'] = df['Age'].round(1)  # Round to 1 decimal place 

In [43]:
df.dtypes

Name               object
Nickname           object
Weight:             int64
Age               float64
SLpM:             float64
Str. Acc.:         object
SApM:             float64
Str. Def:          object
TD Avg.:          float64
TD Acc.:           object
TD Def.:           object
Sub. Avg.:        float64
Height:            object
Reach:              int64
Wins                int64
Losses              int64
Draws               int64
Win Percentage    float64
dtype: object

In [44]:
# Remove percentage signs from accuracy/defense columns
percentage_cols = ["Str. Acc.:", "Str. Def:", "TD Acc.:", "TD Def.:"]
for col in percentage_cols:
    if col in df.columns:
        df[col] = df[col].replace({'%': ''}, regex=True)

In [45]:
df

,Name,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage
0,Tom Aaron,No Name,155,47.5,0.00,0,0.00,0,0.00,0,0,0.0,NaN,0,5,3,0,0.62
1,Danny Abbadi,The Assassin,155,42.5,3.29,38,4.41,57,0.00,0,77,0.0,"5' 11""",0,4,6,0,0.40
2,Nariman Abbasov,Bayraktar,155,31.9,3.00,20,5.67,46,0.00,0,66,0.0,"5' 8""",66,28,4,0,0.88
3,Darion Abbey,No Name,265,32.9,8.44,50,14.06,28,0.00,0,0,0.0,"6' 2""",80,9,5,0,0.64
4,David Abbott,Tank,265,60.7,1.35,30,3.55,38,1.07,33,66,0.0,"6' 0""",0,10,15,0,0.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4439,Alex Zuniga,No Name,145,NaN,0.00,0,0.00,0,0.00,0,0,0.0,NaN,0,6,3,0,0.67
4440,George Zuniga,No Name,185,NaN,7.64,38,5.45,37,0.00,0,100,0.0,"5' 9""",0,3,1,0,0.75
4441,Allan Zuniga,Tigre,155,33.8,3.93,52,1.80,61,0.00,0,57,1.0,"5' 7""",70,13,1,0,0.93
4442,Brahyam Zurcher,El Jarretas,145,29.8,1.86,15,5.57,59,0.00,0,0,0.0,"5' 10""",70,9,1,0,0.90


In [46]:
df.dtypes

Name               object
Nickname           object
Weight:             int64
Age               float64
SLpM:             float64
Str. Acc.:         object
SApM:             float64
Str. Def:          object
TD Avg.:          float64
TD Acc.:           object
TD Def.:           object
Sub. Avg.:        float64
Height:            object
Reach:              int64
Wins                int64
Losses              int64
Draws               int64
Win Percentage    float64
dtype: object

In [47]:
df['Str. Acc.:'] = df['Str. Acc.:'].astype(float)
df['Str. Acc.:'] = df['Str. Acc.:']/100
df['Str. Def:'] = df['Str. Def:'].astype(float)
df['Str. Def:'] = df['Str. Def:']/100
df['TD Acc.:'] = df['TD Acc.:'].astype(float)
df['TD Acc.:'] = df['TD Acc.:']/100
df['TD Def.:'] = df['TD Def.:'].astype(float)
df['TD Def.:'] = df['TD Def.:']/100

In [48]:
df

,Name,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage
0,Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,5,3,0,0.62
1,Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,4,6,0,0.40
2,Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,28,4,0,0.88
3,Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,9,5,0,0.64
4,David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,10,15,0,0.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4439,Alex Zuniga,No Name,145,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,6,3,0,0.67
4440,George Zuniga,No Name,185,NaN,7.64,0.38,5.45,0.37,0.00,0.00,1.00,0.0,"5' 9""",0,3,1,0,0.75
4441,Allan Zuniga,Tigre,155,33.8,3.93,0.52,1.80,0.61,0.00,0.00,0.57,1.0,"5' 7""",70,13,1,0,0.93
4442,Brahyam Zurcher,El Jarretas,145,29.8,1.86,0.15,5.57,0.59,0.00,0.00,0.00,0.0,"5' 10""",70,9,1,0,0.90


In [49]:
# Save cleaned data
df.to_csv('fighters_clean.csv', index=False)
print(f"Saved {len(df)} cleaned fighters to fighters_clean.csv")
print(f"\nFinal columns: {df.columns.tolist()}")
df.head()

Saved 4444 cleaned fighters to fighters_clean.csv

Final columns: ['Name', 'Nickname', 'Weight:', 'Age', 'SLpM:', 'Str. Acc.:', 'SApM:', 'Str. Def:', 'TD Avg.:', 'TD Acc.:', 'TD Def.:', 'Sub. Avg.:', 'Height:', 'Reach:', 'Wins', 'Losses', 'Draws', 'Win Percentage']


,Name,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage
0,Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,5,3,0,0.62
1,Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,4,6,0,0.40
2,Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,28,4,0,0.88
3,Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,9,5,0,0.64
4,David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,10,15,0,0.40
